






# Airbnb Calendar- Limpieza y análisis exploratorio

## Análisis del Dataset Calendar – Airbnb Barcelona 2025

### 1. Introducción

El presente notebook tiene como objetivo analizar el dataset **calendar** correspondiente a la ciudad de Barcelona para el año 2025, obtenido a través de la plataforma Inside Airbnb.

Este dataset contiene información diaria sobre la disponibilidad y el precio de cada anuncio publicado en Airbnb, permitiendo estudiar la dinámica temporal de la oferta en el mercado de alquiler turístico.

El análisis busca:

- Comprender la estructura del dataset y sus principales variables.
- Identificar patrones temporales en la disponibilidad de los anuncios.
- Analizar el comportamiento estratégico de los hosts en relación con la gestión de la disponibilidad.
- Detectar posibles tendencias estacionales.
- Preparar y transformar los datos para su posterior almacenamiento en una base de datos SQL.

### 2. Contexto del dataset

El archivo `calendar.csv` surge de un proceso de recolección de datos (*scraping*) llevado a cabo por Inside Airbnb (https://insideairbnb.com/get-the-data/), realizado los días 14 y 15 de septiembre de 2025, sobre los anuncios activos en la plataforma Airbnb en Barcelona.

Cada fila del dataset representa la información correspondiente a un anuncio en una fecha específica, indicando si estuvo disponible o no durante ese día dentro de un horizonte temporal de 365 días.

  

In [1]:

import pandas as pd
import numpy as np
pd.set_option("display.max_rows", None)


### 3. Carga del dataset

Se carga el archivo `calendar.csv`, que contiene información a nivel de calendario anual.

La información surge de un *scrapeo* de datos, realizado por Inside Airbnb, obtenido los días **14 y 15 de septiembre de 2025** a partir de anuncios publicados en la plataforma **Airbnb**.  

Cada fila representa el estado diario de un anuncio (disponible o no disponible) a lo largo de los **365 días del año**.
````}


In [2]:
calendar = pd.read_csv('calendar.csv')


### 4. Exploración inicial del dataset
En este paso se revisa la estructura general del dataset, el número de registros, los tipos de datos y la presencia de valores nulos.
Este análisis permite identificar posibles problemas de calidad de datos y definir los pasos de limpieza necesarios.

- Cantidad total de filas previo a la limpieza: *7.084.654* ( total de anuncios durante los 365 días del periodo septiembre 2025- septiembre 2026)
- Cantidad de columnas previo a la limpieza: 7
- Las columnas 'precio' y 'precio ajustado' presentan valores nulos
  

In [3]:
calendar.shape

(7084654, 7)

In [4]:
calendar.head()

,listing_id,date,available,price,adjusted_price,minimum_nights,maximum_nights
0,18674,2025-09-15,f,NaN,NaN,3,999
1,18674,2025-09-16,t,NaN,NaN,2,999
2,18674,2025-09-17,t,NaN,NaN,2,999
3,18674,2025-09-18,t,NaN,NaN,2,999
4,18674,2025-09-19,f,NaN,NaN,3,999


In [5]:
## Saber la cantidad de columnas y los tipos de datos de cada una: 
calendar.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7084654 entries, 0 to 7084653
Data columns (total 7 columns):
 #   Column          Dtype  
---  ------          -----  
 0   listing_id      int64  
 1   date            object 
 2   available       object 
 3   price           float64
 4   adjusted_price  float64
 5   minimum_nights  int64  
 6   maximum_nights  int64  
dtypes: float64(2), int64(3), object(2)
memory usage: 378.4+ MB


In [6]:
##  Cantidad total de filas:  7.084.654 . Es el total de los anuncios durante los 365 días 
calendar.describe(include='all')

,listing_id,date,available,price,adjusted_price,minimum_nights,maximum_nights
count,7.084654e+06,7084654,7084654,0.0,0.0,7.084654e+06,7.084654e+06
unique,NaN,366,2,NaN,NaN,NaN,NaN
top,NaN,2025-09-15,t,NaN,NaN,NaN,NaN
freq,NaN,19410,3788781,NaN,NaN,NaN,NaN
mean,6.210978e+17,NaN,NaN,NaN,NaN,1.932952e+01,2.206370e+05
std,5.893160e+17,NaN,NaN,NaN,NaN,5.097000e+01,2.173785e+07
min,1.867400e+04,NaN,NaN,NaN,NaN,1.000000e+00,1.000000e+00
25%,3.048961e+07,NaN,NaN,NaN,NaN,2.000000e+00,3.300000e+02
50%,7.056056e+17,NaN,NaN,NaN,NaN,4.000000e+00,3.650000e+02
75%,1.181666e+18,NaN,NaN,NaN,NaN,3.100000e+01,1.125000e+03


## 5. Rango total de fecha de todo el dataset
- Fecha mínima: 14/09/2025
- Fecha máxima: 14/09/2026

In [7]:
calendar['date'] = pd.to_datetime(calendar['date'], format='%Y-%m-%d', errors='coerce')

# 3️⃣ Verificar que la conversión se realizó correctamente
print(calendar.dtypes)
print(calendar['date'].head())

listing_id                 int64
date              datetime64[ns]
available                 object
price                    float64
adjusted_price           float64
minimum_nights             int64
maximum_nights             int64
dtype: object
0   2025-09-15
1   2025-09-16
2   2025-09-17
3   2025-09-18
4   2025-09-19
Name: date, dtype: datetime64[ns]


In [8]:
# Rango global de fechas en calendar
fecha_min = calendar['date'].min()
fecha_max = calendar['date'].max()

print(f"Fecha mínima del dataset: {fecha_min}")
print(f"Fecha máxima del dataset: {fecha_max}")

Fecha mínima del dataset: 2025-09-14 00:00:00
Fecha máxima del dataset: 2026-09-14 00:00:00


### 6. Manejo de columnas vacías

In [9]:
## Observamos que 'precio' y 'precio_ajustado' presentan números nulos 
calendar.isna().sum().sort_values(ascending=False)


price             7084654
adjusted_price    7084654
listing_id              0
available               0
date                    0
minimum_nights          0
maximum_nights          0
dtype: int64

In [10]:
#   Eliminación de columnas de precio (no útiles)
calendar.drop(columns=['price', 'adjusted_price'], inplace=True, errors='ignore')

In [11]:
calendar.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7084654 entries, 0 to 7084653
Data columns (total 5 columns):
 #   Column          Dtype         
---  ------          -----         
 0   listing_id      int64         
 1   date            datetime64[ns]
 2   available       object        
 3   minimum_nights  int64         
 4   maximum_nights  int64         
dtypes: datetime64[ns](1), int64(3), object(1)
memory usage: 270.3+ MB


### 7. Limpieza de datos

In [12]:

## Renombrado de columnas 
calendar = calendar.rename(columns={
    'listing_id': 'id_anuncio',
    'date': 'fecha',
    'available': 'disponible',
    'minimum_nights': 'noches_minimas',
    'maximum_nights': 'noches_maximas'
})


### 8. Convertir tipos a formatos compatibles con MySQL

In [13]:
# Convertir datetime a string en formato YYYY-MM-DD
calendar['fecha'] = calendar['fecha'].dt.strftime('%Y-%m-%d')

# Verificar el cambio
print(calendar.dtypes)
print(calendar['fecha'].head())


id_anuncio         int64
fecha             object
disponible        object
noches_minimas     int64
noches_maximas     int64
dtype: object
0    2025-09-15
1    2025-09-16
2    2025-09-17
3    2025-09-18
4    2025-09-19
Name: fecha, dtype: object


In [14]:
print(calendar["disponible"].unique())

['f' 't']


####  8.1 Conversión de la columna `disponible` a tipo entero (`INT`)

La columna `disponible` almacena valores booleanos en formato de texto (`'t'` / `'f'`), lo cual no es compatible de forma nativa con MySQL. Para garantizar una lectura e interpretación correctas, se realiza la siguiente conversión:

| Valor original | Significado   | Valor convertido (`INT`) |
|:--------------:|:-------------:|:------------------------:|
| `'t'`          | Disponible    | `1`                      |
| `'f'`          | No disponible | `0`                      |

> **Criterio de conversión:** se preserva la semántica booleana original, mapeando `'t'` → `1` y `'f'` → `0`.

In [15]:
calendar['disponible'] = calendar['disponible'].map({'t': 1, 'f': 0})

In [16]:
calendar.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7084654 entries, 0 to 7084653
Data columns (total 5 columns):
 #   Column          Dtype 
---  ------          ----- 
 0   id_anuncio      int64 
 1   fecha           object
 2   disponible      int64 
 3   noches_minimas  int64 
 4   noches_maximas  int64 
dtypes: int64(4), object(1)
memory usage: 270.3+ MB


In [17]:
# Explorar los datos antes de tocar nada : 
print(calendar["noches_minimas"].describe())
print(calendar["noches_maximas"].describe())

count    7.084654e+06
mean     1.932952e+01
std      5.097000e+01
min      1.000000e+00
25%      2.000000e+00
50%      4.000000e+00
75%      3.100000e+01
max      1.124000e+03
Name: noches_minimas, dtype: float64
count    7.084654e+06
mean     2.206370e+05
std      2.173785e+07
min      1.000000e+00
25%      3.300000e+02
50%      3.650000e+02
75%      1.125000e+03
max      2.147484e+09
Name: noches_maximas, dtype: float64


 ### 9. Ajustar valores extremos en los límites de "noches_minimas" y "noches_maximas"

In [18]:
# Ajustar valores extremos a límites razonables
# noches_minimas: cap en 365 (máximo 1 año tiene sentido en Airbnb)
calendar.loc[calendar["noches_minimas"] > 365, "noches_minimas"] = 365

# noches_maximas: cap en 3650 (el valor 2^31-1 indica "sin límite" en el sistema original)
calendar.loc[calendar["noches_maximas"] > 3650, "noches_maximas"] = 3650

# Corregir casos donde noches_minimas > noches_maximas (swap)
mask = calendar["noches_minimas"] > calendar["noches_maximas"]
calendar.loc[mask, ["noches_minimas", "noches_maximas"]] = (
    calendar.loc[mask, ["noches_maximas", "noches_minimas"]].values
)

# Noches mínimas > 365
print("Filas con noches_minimas > 365:", (calendar["noches_minimas"] > 365).sum())

# Noches máximas > 3650
print("Filas con noches_maximas > 3650:", (calendar["noches_maximas"] > 3650).sum())

# Noches mínimas > noches máximas
print("Filas con noches_minimas > noches_maximas:", (calendar["noches_minimas"] > calendar["noches_maximas"]).sum())

Filas con noches_minimas > 365: 0
Filas con noches_maximas > 3650: 0
Filas con noches_minimas > noches_maximas: 0


In [19]:
calendar['noches_minimas'] = pd.to_numeric(calendar['noches_minimas'], errors='coerce')
calendar['noches_maximas'] = pd.to_numeric(calendar['noches_maximas'], errors='coerce')

In [20]:
# Revisar tipos
print(calendar.dtypes)

id_anuncio         int64
fecha             object
disponible         int64
noches_minimas     int64
noches_maximas     int64
dtype: object


In [21]:
## Luego de la limpieza se mantuvieron la misma cantidad de filas
calendar.shape

(7084654, 5)

### 10. Eliminación de duplicados en las columnas id_anuncio y fecha

In [22]:

calendar = calendar.drop_duplicates(subset=["id_anuncio", "fecha"])

In [23]:
calendar.shape

(7084654, 5)

### 11. Columnas en minúsculas y sin espacios 

In [24]:
calendar.columns = [c.lower().replace(" ", "_") for c in calendar.columns]

### 12. Resumen del dataset calendar limpio

In [25]:

print("Resumen del dataset limpio:")
calendar.info()

Resumen del dataset limpio:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7084654 entries, 0 to 7084653
Data columns (total 5 columns):
 #   Column          Dtype 
---  ------          ----- 
 0   id_anuncio      int64 
 1   fecha           object
 2   disponible      int64 
 3   noches_minimas  int64 
 4   noches_maximas  int64 
dtypes: int64(4), object(1)
memory usage: 270.3+ MB


In [26]:
calendar.head()

,id_anuncio,fecha,disponible,noches_minimas,noches_maximas
0,18674,2025-09-15,0,3,999
1,18674,2025-09-16,1,2,999
2,18674,2025-09-17,1,2,999
3,18674,2025-09-18,1,2,999
4,18674,2025-09-19,0,3,999


### 13. Filtrado de calendar según anuncios activos en listings_clean

Filtrado de anuncios activos: 
- Se filtró calendar para quedarse solo con los anuncios que aparecen en listings_clean.

- Se validó que ambos datasets tengan la misma cantidad de anuncios activos (16,676).

Resultado: calendar_filtrado y listings_clean contienen los mismos anuncios, listos para análisis y resúmenes posteriores.

In [27]:
import pandas as pd

# Cargar el archivo listings_clean
listings_clean = pd.read_csv("listings_clean.csv") 

In [28]:
# Ver nombres exactos de columnas
print(list(listings_clean.columns))

['id_anuncio', 'id_host', 'nombre_host', 'cantidad_anuncios_host', 'cantidad_total_anuncios_host', 'barrio', 'latitud', 'longitud', 'tipo_alojamiento', 'capacidad', 'precio', 'noches_minimas', 'noches_maximas', 'puntuacion_general', 'puntuacion_ubicacion', 'puntuacion_valor', 'licencia', 'last_scraped', 'precio_por_persona', 'tipo_host', 'tipo_licencia']


In [29]:
# Filtrar calendar según los anuncios que están en listings_clean
calendar_filtrado = calendar[calendar['id_anuncio'].isin(listings_clean['id_anuncio'].unique())]

# Validación
print("Cantidad de anuncios en listings_clean:", listings_clean['id_anuncio'].nunique())
print("Cantidad de anuncios en calendar_filtrado:", calendar_filtrado['id_anuncio'].nunique())

# IDs que no coinciden 
set_calendar = set(calendar_filtrado['id_anuncio'].unique())
set_listings = set(listings_clean['id_anuncio'].unique())

print("IDs en calendar pero no en listings_clean:", set_calendar - set_listings)
print("IDs en listings_clean pero no en calendar:", set_listings - set_calendar)

Cantidad de anuncios en listings_clean: 16676
Cantidad de anuncios en calendar_filtrado: 16676
IDs en calendar pero no en listings_clean: set()
IDs en listings_clean pero no en calendar: set()


In [30]:
# Contar cantidad de registros por anuncio
dias_por_anuncio = calendar.groupby('id_anuncio')['fecha'].count().reset_index()
dias_por_anuncio.rename(columns={'fecha':'dias_registrados'}, inplace=True)

# Revisar si hay anuncios con más de 365 o menos de 365 días
dias_extra = dias_por_anuncio[dias_por_anuncio['dias_registrados'] != 365]

print("Anuncios con días distintos a 365:")
print(dias_extra)

# Resumen rápido
print("\nCantidad de anuncios con menos de 365 días:", (dias_por_anuncio['dias_registrados'] < 365).sum())
print("Cantidad de anuncios con más de 365 días:", (dias_por_anuncio['dias_registrados'] > 365).sum())

Anuncios con días distintos a 365:
                id_anuncio  dias_registrados
5972              38768158               366
9342    660521529878840414               366
10508   826043947275498945               366
16765  1370855426145980536               366

Cantidad de anuncios con menos de 365 días: 0
Cantidad de anuncios con más de 365 días: 4


In [31]:
calendar_filtrado.shape

(6086742, 5)

## 14. Observación sobre días extra en el dataset `calendar`

Al calcular el número de registros por anuncio, se esperaba que cada anuncio tuviera exactamente 365 días (del 14/09/2025 al 14/09/2026):

- **Total esperado:** 16,676 anuncios × 365 días = 6,086,740 filas  
- **Total real:** 6,086,742 filas → +2 filas de diferencia  

### Causa de la discrepancia
- 4 anuncios presentan **366 días registrados** en lugar de 365:

| id_anuncio               | días_registrados |
|---------------------------|----------------|
| 38768158                 | 366            |
| 660521529878840414       | 366            |
| 826043947275498945       | 366            |
| 1370855426145980536      | 366            |

- Ningún anuncio tiene menos de 365 días.  
- Estos días extra probablemente corresponden a registros adicionales en el calendario (inicio o fin de disponibilidad) o duplicados leves.

### Interpretación
- La diferencia de filas **no indica un error**, sino un comportamiento normal en datos reales.  
- Para análisis general, los 2 días extra **no afectan significativamente** las métricas globales.  

 **Mantener los días extra:** conserva toda la información disponible de cada anuncio.  


### 15. Análisis de Outliers

In [32]:
import numpy as np

# Columnas numéricas a revisar
cols = ['noches_minimas','noches_maximas']

for col in cols:
    Q1 = calendar[col].quantile(0.25)
    Q3 = calendar[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = calendar[(calendar[col] < lower_bound) | (calendar[col] > upper_bound)]
    print(f"\nColumna: {col}")
    print(f"Rango intercuartil: {Q1} - {Q3}")
    print(f"Outliers detectados: {outliers.shape[0]}")
    print(outliers[[col]].describe())


Columna: noches_minimas
Rango intercuartil: 2.0 - 31.0
Outliers detectados: 135133
       noches_minimas
count   135133.000000
mean       199.534192
std        121.932683
min         75.000000
25%         90.000000
50%        120.000000
75%        365.000000
max        365.000000

Columna: noches_maximas
Rango intercuartil: 330.0 - 1125.0
Outliers detectados: 726
       noches_maximas
count           726.0
mean           3650.0
std               0.0
min            3650.0
25%            3650.0
50%            3650.0
75%            3650.0
max            3650.0


### 16. Análisis Exploratorio – Outliers en `calendar`

Se detectaron outliers en las columnas relacionadas con las noches de estadía:

### 1️⃣ Noches mínimas (`noches_minimas`)
- **Rango intercuartil:** 2 – 31 noches  
- **Outliers detectados:** 135,133 filas (~2% del dataset)  
- **Valores extremos:** 75, 90, 120, hasta 365 noches  
- **Interpretación:**  
  - Representan anuncios con estadías largas o incluso anuales.  
  - Son significativamente mayores que la mayoría de los anuncios.  
  - Pueden distorsionar métricas promedio de disponibilidad si se incluyen en el análisis general.

### 2️⃣ Noches máximas (`noches_maximas`)
- **Rango intercuartil:** 330 – 1,125 noches  
- **Outliers detectados:** 726 filas  
- **Valor extremo común:** 3,650 noches  
- **Interpretación:**  
  - Valores por defecto que indican “sin límite real” de noches máximas.  
  - Son casos extremos y claramente atípicos respecto al rango habitual.



In [33]:
calendar_filtrado.shape

(6086742, 5)

## 17.Conclusión del análisis del dataset `Calendar` – Airbnb Barcelona 2025-2026

El dataset contiene información diaria de disponibilidad de cada anuncio desde el **14/09/2025 hasta el 14/09/2026**.

Se realizó la limpieza y preparación de los datos:

- Conversión de tipos: `fecha` de object a datetime para concer el rango de disponibilidad diaria del dataset.  
- Limpieza de noches mínimas y máximas, ajustando valores extremos y corrigiendo inconsistencias.  
- Eliminación de duplicados y columnas vacías (`price`, `adjusted_price`), utilizando la columna `precio` de `listings_clean` como referencia.  

Se analizaron las fechas mínimas y máximas de cada anuncio, identificando el rango de días que cada anuncio estuvo disponible o registrado en el dataset.

Luego se cargó `listings_clean` y se filtró `calendar` para quedarse solo con los anuncios que aparecen en este dataset, asegurando consistencia entre ambos.

Se validó que ambos datasets contengan la misma cantidad de anuncios activos: **16,676**.

### Observaciones sobre filas y días
- Se esperaba que `calendar_filtrado` tuviera **16,676 × 365 = 6,086,740 filas**, pero se registraron **6,086,742 filas**.  
- La diferencia se debe a **4 anuncios que tienen 366 días registrados** en lugar de 365, probablemente por registros adicionales en inicio o fin de disponibilidad o duplicados leves.  
- Ningún anuncio tiene menos de 365 días.  
- Estos días extra **no afectan significativamente** las métricas globales, y se recomienda mantenerlos para no perder información.  

### Outliers en noches
- **Noches mínimas:** 135,133 outliers (valores ≥75 noches, hasta 365) representan anuncios con estadías largas o anuales.  
- **Noches máximas:** 726 outliers (valor = 3,650 noches) indican límites máximos por defecto, claramente atípicos.  
- Para análisis general, se recomienda **marcarlos o filtrarlos**, y para análisis especial mantenerlos como casos excepcionales.  

### Adaptación de columnas para MySQL
- Se adaptaron los tipos de columnas para que sean compatibles con MySQL:  
  - `id_anuncio`, `disponible`, `noches_minimas`, `noches_maximas` → **INT**  
  - `fecha` → **string (`object`) en formato `YYYY-MM-DD`** para que MySQL lo interprete como DATE al cargar el CSV.  
- Esto asegura que el dataset sea **compatible directamente con `LOAD DATA`**, manteniendo consistencia y permitiendo análisis posteriores en MySQL.  

Como resultado, `calendar_filtrado` quedó con **6,086,742 filas y 5 columnas**, listo para análisis exploratorio, métricas de ocupación, patrones de disponibilidad y visualizaciones por anuncio.

## 18.  Guardar dataset limpio en CSV

In [34]:

calendar_filtrado.to_csv("calendar_limpio_filtrado.csv", index=False)
